# PART 6. 추천 시스템 (유사도 기반)

## 수업 목표
- 상품 카테고리를 숫자로 변환하고 코사인 유사도로 비슷한 상품을 추천합니다.
- 원핫 인코딩과 코사인 유사도의 개념을 이해합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 상품 카테고리 정보만 사용합니다. |
| 코드 관점 | get_dummies(), cosine_similarity() 사용법을 익힙니다. |
| AI 관점 | 이 단계는 딥러닝 임베딩 이전의 "유사도 기반 추천 기초"입니다. |
| 강사 메모 | 데이터가 많으면 메모리 부족이 발생하므로 3000개만 사용합니다. |

---


## 📌 이 파트에서 사용하는 API 흐름 (scikit-learn)

| 단계 | 함수 | 역할 |
|---|---|---|
| ① | cosine_similarity() | 유사도 계산 |
| ② | recommend_product() | 추천 함수 실행 |

> 추천 시스템은 fit/predict 구조가 아닌 유사도 계산 방식입니다.


## 흐름
```
데이터 준비 → 원핫 인코딩(문자→숫자) → 코사인 유사도 계산 → 추천 함수 → 저장
```

## 핵심 개념

| 개념 | 설명 |
|---|---|
| 원핫 인코딩 | "auto" → auto 컬럼만 1, 나머지는 0으로 변환 |
| 코사인 유사도 | 두 상품이 얼마나 비슷한지 0~1 점수로 계산 |
| 유사도 1.0 | 완전히 같은 상품 (자기 자신) |
| 유사도 0.0 | 완전히 다른 상품 |

In [11]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# 셀 2. 라이브러리 불러오기

import os
import pickle
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
# cosine_similarity: 두 벡터가 얼마나 비슷한지 0~1 점수로 계산하는 함수

base_path = "/content/drive/MyDrive/Olist/data"
model_path = "/content/drive/MyDrive/Olist/model"
os.makedirs(model_path, exist_ok=True)

In [15]:
# 셀 3. 통합 데이터 불러오기

df = pd.read_csv(f"{base_path}/olist_master_data.csv")
print("데이터 크기:", df.shape)

데이터 크기: (119143, 40)


In [16]:
# 셀 4. 추천에 필요한 컬럼만 선택

recommend_df = (
    df[["product_id", "product_category_name_english"]]
    .dropna()            # 카테고리 없는 상품 제거
    .drop_duplicates()   # 동일 상품 중복 제거
    .reset_index(drop=True)
)

print(f"전체 상품 수: {len(recommend_df):,}")

# 실습 속도를 위해 3000개만 사용합니다.
# (전체 사용 시 메모리 부족 발생 가능)
recommend_df = recommend_df.head(3000).reset_index(drop=True)
print(f"실습용 상품 수: {len(recommend_df):,}")
recommend_df.head()

전체 상품 수: 32,328
실습용 상품 수: 3,000


,product_id,product_category_name_english
0,87285b34884572647811a353c7ac498a,housewares
1,595fac2a385ac33a80bd5114aec74eb8,perfumery
2,aa4383b373c6aca5d8797843e5594415,auto
3,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop
4,65266b2da20d04dbe00c5c2d3bb7859e,stationery


In [17]:
# 셀 5. 원핫 인코딩 (문자 카테고리 → 숫자 0/1)
# 예) auto 카테고리이면 → auto 컬럼만 True, 나머지는 False

category_matrix = pd.get_dummies(
    recommend_df["product_category_name_english"]
)

print("원핫 인코딩 결과 크기:", category_matrix.shape)
# ▶ (3000, 62) → 상품 3000개 × 카테고리 62종류

# 일부만 출력 (전체 62컬럼 중 앞 5개만)
category_matrix.iloc[:5, :5]

원핫 인코딩 결과 크기: (3000, 62)


,agro_industry_and_commerce,air_conditioning,art,audio,auto
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,True
3,False,False,False,False,False
4,False,False,False,False,False


## 원핫 인코딩 쉬운 예시

| 상품 | furniture_decor | perfumery | auto |
|---|---|---|---|
| 의자 | 1 | 0 | 0 |
| 향수 | 0 | 1 | 0 |
| 자동차용품 | 0 | 0 | 1 |

같은 카테고리이면 같은 숫자 패턴 → 유사도가 높게 계산됩니다.


### 🔵 ① cosine_similarity() → 유사도 계산


In [18]:
# 셀 6. 코사인 유사도 계산
# 상품 3000개끼리 서로 얼마나 비슷한지 전부 계산합니다.
# 결과: (3000, 3000) 크기의 유사도 행렬

similarity_matrix = cosine_similarity(category_matrix)

print("유사도 행렬 크기:", similarity_matrix.shape)
# ▶ 3000 x 3000 = 상품 3000개 × 상품 3000개 비교표

유사도 행렬 크기: (3000, 3000)


## 유사도 행렬 예시

|  | 상품A | 상품B | 상품C |
|---|---|---|---|
| 상품A | 1.0 | 1.0 | 0.0 |
| 상품B | 1.0 | 1.0 | 0.0 |
| 상품C | 0.0 | 0.0 | 1.0 |

- 1.0 → 같은 카테고리 (매우 비슷)
- 0.0 → 다른 카테고리 (전혀 다름)

In [19]:
# 셀 7. 상품 ID와 행렬 위치(index) 연결
# 상품 ID(긴 문자열)를 숫자 위치(index)로 빠르게 찾기 위한 매핑

product_to_index = pd.Series(
    recommend_df.index,
    index=recommend_df["product_id"]
).drop_duplicates()

product_to_index.head()

,0
product_id,
87285b34884572647811a353c7ac498a,0
595fac2a385ac33a80bd5114aec74eb8,1
aa4383b373c6aca5d8797843e5594415,2
d0b61bfb1de832b15ba9d266ca96e5b0,3
65266b2da20d04dbe00c5c2d3bb7859e,4


In [20]:
# 셀 8. 추천 함수 만들기

def recommend_product(product_id, top_n=5):
    """
    특정 상품과 비슷한 상품 top_n개를 추천합니다.

    매개변수:
        product_id: 기준이 될 상품 ID
        top_n: 추천할 상품 개수 (기본값: 5)
    """
    # 입력한 product_id가 데이터에 없는 경우
    if product_id not in product_to_index:
        return "해당 상품을 찾을 수 없습니다."

    # 상품 ID → 행렬 위치(index) 찾기
    idx = product_to_index[product_id]

    # idx번 상품과 모든 상품의 유사도 점수 가져오기
    scores = list(enumerate(similarity_matrix[idx]))

    # 유사도 높은 순으로 정렬
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # 자기 자신(1위)은 제외하고 top_n개 선택
    scores = scores[1 : top_n + 1]

    # 결과 정리
    result = []
    for i, score in scores:
        pid      = recommend_df.iloc[i]["product_id"]
        category = recommend_df.iloc[i]["product_category_name_english"]
        result.append({
            "카테고리": category,
            "유사도":   round(score, 2)
        })

    return pd.DataFrame(result)


### 🔵 ② recommend_product() → 추천 실행


In [21]:
# 셀 9. 추천 테스트

sample_id       = recommend_df.iloc[0]["product_id"]
sample_category = recommend_df.iloc[0]["product_category_name_english"]

print(f"기준 상품 카테고리: {sample_category}")
print()
recommend_product(sample_id, top_n=5)
# ▶ 같은 카테고리 상품이 추천되면 정상입니다.

기준 상품 카테고리: housewares



,카테고리,유사도
0,housewares,1.0
1,housewares,1.0
2,housewares,1.0
3,housewares,1.0
4,housewares,1.0


In [22]:
# 셀 10. 추천 모델 저장 (STEP2에서 API로 사용)
# pickle: 파이썬 데이터를 파일로 저장/불러오기 하는 라이브러리

with open(f"{model_path}/part6_recommend_model.pkl", "wb") as f:
    pickle.dump({
        "recommend_df":      recommend_df,       # 추천용 상품 목록
        "similarity_matrix": similarity_matrix,  # 상품 간 유사도 표
        "product_to_index":  product_to_index,   # 상품ID ↔ index 매핑
    }, f)

print("추천 모델 저장 완료!")
print("저장 위치:", f"{model_path}/part6_recommend_model.pkl")

추천 모델 저장 완료!
저장 위치: /content/drive/MyDrive/Olist/model/part6_recommend_model.pkl


---

## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 카테고리 유사도로 비슷한 상품을 추천합니다. |
| 현재 한계 | 가격, 리뷰, 구매 이력은 반영 안 됩니다. |
| 발전 방향 | 가격, 리뷰, 설명 텍스트 등을 추가하면 더 정교해집니다. |
| 다음 단계 | PART 7에서 리뷰 텍스트로 감성 분석을 진행합니다. |